In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer


class State(TypedDict):
    topic: str
    joke: str


def generate_joke(state: State):
    writer = get_stream_writer()
    writer({"status": "thinking of a joke..."})
    return {"joke": f"Why did the {state['topic']} go to school? To get a sundae education!"}

graph = (
    StateGraph(State)
    .add_node(generate_joke)
    .add_edge(START, "generate_joke")
    .add_edge("generate_joke", END)
    .compile()
)

In [2]:
for part in graph.stream(
    {"topic": "ice cream"}, # type: ignore
    stream_mode=["values", "updates", "messages", "custom"],
    version="v2",
):
    if part["type"] == "values":
        # ValuesStreamPart — full state snapshot after each step
        print('[Values]:')
        print(f"State: {part['data']}")
    elif part["type"] == "updates":
        # UpdatesStreamPart — only the changed keys from each node
        print('[Updates]:')
        for node_name, state in part["data"].items():
            print(f"Node `{node_name}` updated: {state}")
    elif part["type"] == "messages":
        # MessagesStreamPart — (message_chunk, metadata) from LLM calls
        msg, metadata = part["data"]
        print(msg.content, end="", flush=True)
    elif part["type"] == "custom":
        # CustomStreamPart — arbitrary data from get_stream_writer()
        print('[Custom]:')
        print(f"Status: {part['data']['status']}")

[Values]:
State: {'topic': 'ice cream'}
[Custom]:
Status: thinking of a joke...
[Updates]:
Node `generate_joke` updated: {'joke': 'Why did the ice cream go to school? To get a sundae education!'}
[Values]:
State: {'topic': 'ice cream', 'joke': 'Why did the ice cream go to school? To get a sundae education!'}
